# Data

In [1]:
!ls

games-all.json				proof-of-concept-round3-backup3.ipynb
log.local.bin				proof-of-concept-round3.ipynb
proof-of-concept.ipynb			proof-of-concept-round3-parallel2.ipynb
proof-of-concept-round2-backup-1.ipynb	proof-of-concept-round3-parallel.ipynb
proof-of-concept-round2.ipynb		proof-of-concept-round4.ipynb
proof-of-concept-round3-backup1.ipynb	stand
proof-of-concept-round3-backup2.ipynb


In [2]:
!tail stand/log.local.txt -n 3

191069 0.001350080827
183391 0.001279927092
183504 0.00114216411


In [3]:
!wc -l stand/log.local.txt

112073874 stand/log.local.txt


In [1]:
import collections
import pickle
import numpy as np
import tqdm
import os
import gc


def load(limit, raw_path = "stand/log.local.txt", path = "log.local.bin", key_games = None, seed = 17, det_attempts = 0):
    readvector = lambda s : np.array(list(map(float, s.strip()[1:-2].split(","))))
    requests = list()
    docembs = collections.defaultdict(dict)

    if os.path.isfile(path):
        with open(path, "rb") as f:
            flimit, frequests, fdocembs = pickle.load(f)
            if flimit == limit:
                requests, docembs = frequests, fdocembs
            else:
                print(f"WARN: buffered limit is different, {flimit} != {limit}, reloading...")

    if not requests:
        with open(raw_path) as f:
            req = list()
            reqid = None
            models = list()
            prevreqmodel = None
            reqmodel = dict()
            prevmodelid = -1
            bannermodelid = -1
            for i, line in tqdm.tqdm_notebook(enumerate(f)):
                if line.startswith("Model = 6;"):
                    prevreqmodel = reqmodel
                    reqmodel = dict()

                if line.startswith("Model = "):
                    spl = line.split(" ")
                    prevmodelid = int(spl[2][:-1])
                    bannermodelid = max(bannermodelid , prevmodelid)
                    reqmodel[prevmodelid] = readvector(spl[3])
                elif line.startswith("dbid"):
                    spl = line.split(" ")
                    dbid = int(spl[1][:-1])
                    docembs[bannermodelid][dbid] = readvector(spl[2])
                elif line.startswith("seed"):
                    if len(requests) >= limit:
                        break
                    if req:
                        requests.append((reqid, prevreqmodel, sorted(req)))
                        req = list()
                    reqid = "$_" + (line.split()[1] + "_" + line.split()[3])
                else:
                    req.append(
                        (int(line.split()[0]), float(line.split()[1]))
                    )
        
        with open(path, "wb") as f:
            pickle.dump((limit, requests, docembs), f)

    games_count = len(requests[0][2])
    assert games_count == 16514
    requests = [r for r in requests if len(r[2]) == games_count]
    
    print([(i, len(docembs[i].keys())) for i in docembs])  # should be equal
    docblocks = {
        mid : np.array([x[1] for x in sorted(list(docembs[mid].items()))])
        for mid in docembs
    }
    
    class EvalContext:
        def __init__(self, games_count = games_count, key_size = 100, train_size = 0.7, key_games = None, seed = 17, det_attempts = 0):
            self.games_count = games_count
            self.key_games = (
                np.random.choice(np.arange(games_count), key_size, replace=False)
                if key_games is None else
                key_games
            )

            self.requests = requests
            np.random.seed(seed)
            np.random.shuffle(self.requests)
            
            self.try_det_attempts(det_attempts)

            self.key_reqs = self.requests[:key_size + 1]
            self.key_reqs_idx = np.arange(key_size + 1)

            self.train_split = int(len(self.requests) * train_size)

            assert key_size + 1 < self.train_split

            self.train_reqs = self.requests[key_size + 1: self.train_split]
            self.test_reqs = self.requests[self.train_split:]

            self.slices = ["key", "train", "test"]
            print(len(self.key_reqs), len(self.train_reqs), len(self.test_reqs))

            self.docblocks = docblocks
            
        def get_top_games(self):
            if not hasattr(self, "top_games"):
                embed_games = np.array([
                    np.array([r[2][g_i][1] for r in self.get_requests("train")])
                    for g_i in range(self.games_count)
                ])

                self.embed_games_mean = embed_games.mean(axis=1)
                self.top_games_all = (-self.embed_games_mean).argsort()
                self.top_games = self.top_games_all[:len(self.key_games)]

            return self.top_games
        
        def set_top_games_as_key(self):
            self.key_games = self.get_top_games()
            return self
        
        def try_det_attempts(self, det_attempts, model_id = 6):
            def get_det(r, r_i_array):
                kr = np.array([
                    r[r_i][1][model_id]
                    for r_i in r_i_array[:100]
                ])
                return np.abs(np.linalg.det(kr[:kr.shape[1], :]))

            best_i_array = np.arange(len(self.requests))

            for _ in range(det_attempts):
                # print("try update key_reqs...")
                
                r_i_array = np.arange(len(self.requests))
                np.random.shuffle(r_i_array)
                
                n, o = get_det(self.requests, r_i_array), get_det(self.requests, best_i_array)
                # print(n, o)
                if n > o:
                    best_i_array = r_i_array
                    # print("updated!")

            print("Best det = ", get_det(self.requests, best_i_array))
            
            new_requests = [
                self.requests[i]
                for i in best_i_array
            ]
            
            del self.requests
            gc.collect()

            self.requests = new_requests
            print(get_det(self.requests, np.arange(len(self.requests))))

        def get_requests(self, t = "train"):
            if t == "train":
                return self.train_reqs
            elif t == "key":
                return self.key_reqs
            elif t == "test":
                return self.test_reqs
            else:
                assert False

    return EvalContext(key_games = key_games, seed = seed, det_attempts = det_attempts)

# Custom preparations

In [2]:
import json

def get_cat_top(ctx, games_all_json = "games-all.json", max_per_category = 4):
    with open(games_all_json) as f:
        games_all = json.loads(f.read())

    game2category = {
        g_i["appID"] : g_i["categoryName"]["categories"]
        for g_i in games_all
    }


    flat_game_id_2_app_id = {i: row[0] for i, row in enumerate(ctx.requests[0][2])}

    all_category = set()
    for i, c in game2category.items():
        for c_i in c:
            all_category.add(c_i)

    def get_cat(top_games):
        flat_game_id_2_app_id = {i: row[0] for i, row in enumerate(ctx.requests[0][2])}
        top_cat = set()
        for g_i in top_games:
            app_id = flat_game_id_2_app_id[g_i]
            if app_id in game2category:
                for c_i in game2category[app_id]:
                    top_cat.add(c_i)
            else:
                # print("nf")
                pass
        return top_cat

    cat_top = list()
    cobs = collections.defaultdict(int)

    top_games = ctx.get_top_games()

    for g_i in ctx.top_games_all:
        app_id = flat_game_id_2_app_id[g_i]
        categories = tuple(sorted(game2category[app_id] if app_id in game2category else []))

        skip = False
        for cs_i in categories:
            if cobs[cs_i] >= max_per_category:
                skip = True
                break

        if skip:
            continue

        for cs_i in categories:
            cobs[cs_i] += 1

        cat_top.append(g_i)
        if len(cat_top) == len(top_games):
            break

    return cat_top

# Models

In [3]:
class Popular:
    def __init__(self, ctx):
        self.ctx = ctx
        self.game_avg_scores = {
            t : self.get_user_scores(t).mean(axis = 0)
            for t in self.ctx.slices
        }
        
    def get_user_scores(self, t):
        return np.array([
            np.array([g_i[1] for g_i in r[2]])
            for r in self.ctx.get_requests(t)
        ])
    
    def get_user_embs(self, t):
        assert False
    
    def get_game_embs(self):
        assert False

    def fit(self, t = "train"):
        self.top_logits = self.game_avg_scores[t]
        self.top = np.argsort(-self.top_logits)

    def recommend(self, t):
        return [self.top_logits for _ in self.get_user_scores(t)]
    
    def get_score(self, t, topsize = 100):       
        recs = np.array(self.recommend(t))
        trus = self.get_user_scores(t)

        mse = np.mean((recs - trus) ** 2)

        recs = np.argsort(-recs, axis=1)[:, :topsize]
        trus = np.argsort(-trus, axis=1)[:, :topsize]

        ev = lambda rec_, real_ : len((set(rec_).intersection(set(real_)))) 
            
        results = [
            ev(rec, tru) / float(topsize)
            for rec, tru in zip(recs,trus)
        ]

        print("np.mean(results), mse, len(results) = ",
              np.mean(results), mse, len(results))

        return np.mean(results)

In [4]:
np.argsort(np.array([[2, 0, 1], [9, 10, 19]]), axis=1)

array([[1, 2, 0],
       [0, 1, 2]])

In [5]:
import tensorflow as tf
import copy
import tensorflow_ranking as tfr

DEFAULT_FIT_KWARGS = {
    "learning_rate": 0.001,
    "n": 500,
    "c": 5000,
    "train_popbias": False,
    "train_bias": False,
    "verbose": True,
    "train_dssm": False,
    "loss": "mse",
    "ubatch": 1e9,
}

class CBKnnV0(Popular):
    def __init__(self, ctx, fit_kwargs = dict()):
        super().__init__(ctx)
        
        self.fit_kwargs = fit_kwargs
        
        self.embed_users = {
            t : np.array([
                    np.array([r_i[2][i][1] for i in ctx.key_games])
                    for r_i in ctx.get_requests(t)
                ])
            for t in ctx.slices
        }
        self.embed_users_mean = {
            t : self.embed_users[t].mean(axis = 0)
            for t in ctx.slices
        }
        # embed_users = embed_users - embed_users_mean
        print("self.embed_users['train'].shape = ", self.embed_users['train'].shape )
        
        self.embed_games = np.array([
            np.array([r[2][g_i][1] for r in ctx.get_requests("key")])
            for g_i in range(ctx.games_count)
        ])
        
        self.embed_games_mean = self.embed_games.mean(axis = 0)
        
        # embed_games = embed_games - embed_games_mean
        print("self.embed_games.shape = ", self.embed_games.shape)
        
        self.games_top_key = self.embed_games.mean(axis = 1)
        
        self.games2users = np.array([
            self.embed_games[g_i]
            for g_i in ctx.key_games
        ])
        print("self.games2users.shape = ", self.games2users.shape)
        
        self.core_users_scores = np.array([
            np.array([g_i[1] for g_i in r[2]])
            for r in ctx.get_requests("key")
        ])
        print("self.core_users_scores.shape = ", self.core_users_scores.shape)
        
        self.core_users_embs = self.embed_users["key"]
        print("self.core_users_embs.shape = ", self.core_users_embs.shape)
        
        self.ge_users = (
            (self.embed_users["train"].T / self.embed_users["train"].mean(axis=1)).T @ self.games2users
        )
        # ge_users = embed_users @ games2users
        print("self.ge_users.shape = ", self.ge_users.shape)
    
    def __repr__(self):
        return "CbKnn(" + str(self.fit_kwargs) + ")"
        
    # inherited   
    # def get_user_scores(self, t):
    
    def get_user_embs(self, t):
        return (self.embed_users[t] - self.embed_users_mean["train"]) / self.embed_users[t].std(axis=0)
        # return self.embed_users[t]
    
    def get_game_embs(self):
        return (self.embed_games - self.embed_games.mean(axis=0)) / self.embed_games.std(axis=0)
        #return self.embed_games

    def fit(self, **kwargs):
        p = copy.copy(DEFAULT_FIT_KWARGS)
        p.update(self.fit_kwargs)
        p.update(kwargs)

        self.train_bias = p["train_bias"]
        self.train_popbias = p["train_popbias"]
        self.train_dssm = p["train_dssm"]
        self.loss = p["loss"]

        train_user_scores = self.get_user_scores("train")
        train_user_embs = self.get_user_embs("train")
        game_embs = self.get_game_embs()
        
        initializer = tf.keras.initializers.GlorotUniform()
        values = initializer(shape=(train_user_embs.shape[1], game_embs.shape[1]))
        self.W = tf.Variable(values / 100., trainable = True) 
        self.pb = tf.Variable(1., trainable = True) 
        self.b = tf.Variable(0., trainable = True) 
        
        if p["verbose"]:
            print("self.W = ", self.W)
            print("0-loss = ", tf.reduce_mean((train_user_scores - 0) ** 2))
    
        opt =  tf.keras.optimizers.Adam(learning_rate=p["learning_rate"])

        if self.train_dssm:
            self.train_dssm = True

            dim = game_embs.shape[1]
            self.g_dssm = tf.keras.Sequential([
                tf.keras.layers.Dense(dim, activation='relu'),
                tf.keras.layers.Dense(dim, activation=None)
            ])
            self.g_dssm(game_embs)
            
            # dim = train_user_embs.shape[1]
            self.u_dssm = tf.keras.Sequential([
                tf.keras.layers.Dense(dim, activation='relu'),
                tf.keras.layers.Dense(dim, activation=None)
            ])
            self.u_dssm(train_user_embs)
            
        
        for i in range(p["n"]):
            def loss():
                def get_logits_scores(loss_batch = 1e9):
                    game_slice = None
                    
                    ubatch = p["ubatch"]
                    if ubatch >= train_user_scores.shape[0]:
                        train_user_embs_ = train_user_embs
                        scores_ = train_user_scores
                    else:
                        u_slice = np.random.choice(np.arange(train_user_scores.shape[0]), ubatch, replace = True)
                        train_user_embs_ = train_user_embs[u_slice]
                        scores_ = train_user_scores[u_slice]
                    
                    if loss_batch >= game_embs.shape[0]:
                        game_embs_ = game_embs
                        scores_ = scores_
                    else:
                        game_slice = np.random.choice(np.arange(game_embs.shape[0]), loss_batch, replace = True)
                        game_embs_ = game_embs[game_slice]
                        scores_ = scores_[:, game_slice]
                        
                    
                    logits = train_user_embs_ @ self.W @ game_embs_.T

                    if self.train_dssm:
                        logits += self.u_dssm(train_user_embs_) @ tf.transpose(self.g_dssm(game_embs_))
                    
                    if self.train_popbias:
                        if loss_batch >= game_embs.shape[0]:
                            logits += self.pb * self.game_avg_scores["train"]
                        else:
                            logits += self.pb * self.game_avg_scores["train"][game_slice]
                        
                    if self.train_bias:
                        logits += self.b
                        
                    return logits, scores_
                        
                if self.loss == "mse":
                    logits, scores = get_logits_scores()
                    v = tf.reduce_mean((scores - logits) ** 2)
                elif self.loss == "qmse":
                    logits, scores = get_logits_scores()
                    q_mean = scores.mean(axis=1)
                    v = tf.reduce_mean(((scores.T - q_mean).T - logits) ** 2)
                elif self.loss == "ApproxNDCGLoss":
                    while True:
                        logits, scores = get_logits_scores(
                            loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))

                        v = tfr.keras.losses.ApproxNDCGLoss()(
                            scores.astype(np.float32),
                            logits
                        )
                    
                        if not tf.math.is_nan(v).numpy().any():
                            break
                        else:
                            if p["verbose"]:
                                print("nanloss ignored")
                elif self.loss == "softmax":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    v = -tf.reduce_mean(tf.where(
                        (scores.T >= qs.T).T,
                        tf.nn.softmax(logits, axis=1),
                        0
                    ))
                elif self.loss == "softmax_weighted":
                    logits, scores = get_logits_scores(
                        loss_batch = (64 if "loss_batch" not in p else p["loss_batch"]))
                    
                    qs = np.quantile(scores, p["loss_q"], axis=1)
                    v = -tf.reduce_mean(tf.where(
                        (scores.T >= qs.T).T,
                        tf.nn.softmax(logits, axis=1),
                        0
                    ) * scores)
                elif self.loss == "sigmoid_top_100":
                    logits, scores = get_logits_scores()
                    mask = np.argsort(-scores, axis=1) < 100
                    v = -tf.reduce_mean(
                        tf.math.log_sigmoid((2 * mask - 1) * logits)
                    )
                else:
                    assert False
                    
                v += tf.reduce_mean(self.W * self.W) * p["c"]
                
                if p["verbose"]:
                    print(v.numpy())
                
                return v

            weights = list()
            
            weights += [self.W]

            if self.train_dssm:
                weights += self.u_dssm.weights
                weights += self.g_dssm.weights
            
            if self.train_popbias:
                weights += [self.pb]

            if self.train_bias:
                weights += [self.b]   
                
                
            opt.minimize(loss, weights).numpy()
        print("last loss = ", loss())

    def recommend(self, t):
        logits = self.get_user_embs(t) @ self.W @ self.get_game_embs().T
        
        if self.train_dssm:
            logits += self.u_dssm(self.get_user_embs(t)) @ tf.transpose(self.g_dssm(self.get_game_embs()))

        if self.train_popbias:
            logits += self.pb * self.game_avg_scores["train"]       
            
        if self.train_bias:
            logits += self.b
            
        return logits
    
    # inherited
    # def get_score(self, t, topsize = 100):
    
class DssmKnn(CBKnnV0):
    def __init__(self, ctx, modelid, fit_kwargs=dict()):
        super().__init__(ctx, fit_kwargs=fit_kwargs)
        self.modelid = modelid
        self.embed_games = ctx.docblocks[modelid]
        
    def __repr__(self):
        return "DssmKnn(" + str(self.modelid) + "," + str(self.fit_kwargs) + ")"
        
    def get_user_embs(self, t):
        return np.array([r[1][self.modelid] for r in self.ctx.get_requests(t)])
    
    def get_game_embs(self):
        return self.embed_games
    
def ev(mds, logs=None):
    for i in range(len(mds)):
        print("\n\n\n=======")
        print("model = ", mds[i])
        mds[i].fit()
        tr, te = mds[i].get_score("train"), mds[i].get_score("test")
        print(tr, te)
        if logs is not None:
            logs.append((
                repr(mds[i]),
                tr,
                te
            ))

# Evals

skipgramm-loss +1 -k

In [6]:
L = 2000
N = 3000

In [7]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17), fit_kwargs={
        'c': 0.1, 'train_dssm': True,
        'train_popbias': False, 'train_bias': False,
        'verbose': False, 'loss': 'softmax',
        'loss_batch': 128, 'loss_q': 0.98828125,
        'n': N})
])

WARN: buffered limit is different, 5000 != 2000, reloading...


[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  3.043569729389357e-123
3.043569729389357e-123
101 1276 591
self.embed_users['train'].shape =  (1276, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (1276, 101)



model =  CbKnn({'c': 0.1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'softmax', 'loss_batch': 128, 'loss_q': 0.98828125, 'n': 3000})
last loss =  tf.Tensor(-0.006262389, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6429545454545454 7348.7172013982845 1276
np.mean(results), mse, len(results) =  0.6339763113367175 7603.89493207376 591
0.6429545454545454 0.6339763113367175


In [10]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17).set_top_games_as_key(), fit_kwargs={
        'c': 0.1, 'train_dssm': True,
        'train_popbias': False, 'train_bias': False,
        'verbose': False, 'loss': 'softmax',
        'loss_batch': 128, 'loss_q': 0.98828125,
        'n': N})
])

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
3.043569729389357e-123
101 1276 591
self.embed_users['train'].shape =  (1276, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (1276, 101)



model =  CbKnn({'c': 0.1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'softmax', 'loss_batch': 128, 'loss_q': 0.98828125, 'n': 3000})
last loss =  tf.Tensor(-0.0034361267, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6721473354231976 2345.678 1276
np.mean(results), mse, len(results) =  0.6521827411167511 2377.5872 591
0.6721473354231976 0.6521827411167511


In [ ]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 0.1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'softmax',
            'loss_batch': 128, 'loss_q': 0.98828125,
            'n': N
        }
    )
])

"""
=======
model =  CbKnn({'c': 0.1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'softmax', 'loss_batch': 128, 'loss_q': 0.98828125, 'n': 3000})
last loss =  tf.Tensor(-0.0034166388, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6599137931034482 3639.7168 1276
np.mean(results), mse, len(results) =  0.6482064297800338 3873.2979 591
0.6599137931034482 0.6482064297800338
"""

None

In [ ]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000), fit_kwargs={
        'c': 0.1, 'train_dssm': True,
        'train_popbias': False, 'train_bias': False,
        'verbose': False, 'loss': 'softmax',
        'loss_batch': 128, 'loss_q': 0.98828125,
        'n': N})
])

"""
=======
model =  CbKnn({'c': 0.1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'softmax', 'loss_batch': 128, 'loss_q': 0.98828125, 'n': 3000})
last loss =  tf.Tensor(-0.0034408586, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6463479623824452 3839.069 1276
np.mean(results), mse, len(results) =  0.6386802030456853 3777.6387 591
0.6463479623824452 0.6386802030456853
"""

None

In [18]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17), fit_kwargs={
        'c': 0.1, 'train_dssm': True,
        'train_popbias': False, 'train_bias': False,
        'verbose': False, 'loss': 'softmax',
        'loss_batch': 256, 'loss_q': 1- 100./16514,
        'n': N})
])

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
3.043569729389357e-123
101 1276 591
self.embed_users['train'].shape =  (1276, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (1276, 101)



model =  CbKnn({'c': 0.1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'softmax', 'loss_batch': 256, 'loss_q': 0.9939445319123168, 'n': 3000})
last loss =  tf.Tensor(-0.0015841962, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6530094043887147 10169.485 1276
np.mean(results), mse, len(results) =  0.6300676818950931 9776.362 591
0.6530094043887147 0.6300676818950931


In [17]:
256 * (100./16514)

1.5501998304468936

In [19]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17).set_top_games_as_key(), fit_kwargs={
        'c': 0.1, 'train_dssm': True,
        'train_popbias': False, 'train_bias': False,
        'verbose': False, 'loss': 'softmax',
        'loss_batch': 256, 'loss_q': 1- 100./16514,
        'n': N})
])

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
3.043569729389357e-123
101 1276 591
self.embed_users['train'].shape =  (1276, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (1276, 101)



model =  CbKnn({'c': 0.1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'softmax', 'loss_batch': 256, 'loss_q': 0.9939445319123168, 'n': 3000})
last loss =  tf.Tensor(-0.0016657398, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6460579937304075 2364.48 1276
np.mean(results), mse, len(results) =  0.6361928934010153 2367.4382 591
0.6460579937304075 0.6361928934010153


In [ ]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(), fit_kwargs={
        'c': 0.1, 'train_dssm': True,
        'train_popbias': False, 'train_bias': False,
        'verbose': False, 'loss': 'softmax',
        'loss_batch': 256, 'loss_q': 1- 100./16514,
        'n': N})
])

"""

=======
model =  CbKnn({'c': 0.1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'softmax', 'loss_batch': 256, 'loss_q': 0.9939445319123168, 'n': 3000})
last loss =  tf.Tensor(-0.0022031288, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6496081504702195 1470.1129 1276
np.mean(results), mse, len(results) =  0.6495431472081218 1630.1082 591
0.6496081504702195 0.6495431472081218
"""

None

In [23]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17).set_top_games_as_key(), fit_kwargs={
        'c': 0.1, 'train_dssm': True,
        'train_popbias': False, 'train_bias': False,
        'verbose': False, 'loss': 'softmax',
        'loss_batch': 256, 'loss_q': 1- 100./16514,
        'n': N})
])

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
3.043569729389357e-123
101 1276 591
self.embed_users['train'].shape =  (1276, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (1276, 101)



model =  CbKnn({'c': 0.1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'softmax', 'loss_batch': 256, 'loss_q': 0.9939445319123168, 'n': 3000})
last loss =  tf.Tensor(-0.001614709, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6531739811912226 2444.5044 1276
np.mean(results), mse, len(results) =  0.6219458544839256 2619.62 591
0.6531739811912226 0.6219458544839256


In [26]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17).set_top_games_as_key(), fit_kwargs={
        'c': 0.1, 'train_dssm': True,
        'train_popbias': False, 'train_bias': False,
        'verbose': False, 'loss': 'softmax',
        'loss_batch': 256, 'loss_q': 1- 100./16514,
        'n': N})
])

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
3.043569729389357e-123
101 1276 591
self.embed_users['train'].shape =  (1276, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (1276, 101)



model =  CbKnn({'c': 0.1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'softmax', 'loss_batch': 256, 'loss_q': 0.9939445319123168, 'n': 3000})
last loss =  tf.Tensor(-0.0016285101, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6490673981191223 2072.4397 1276
np.mean(results), mse, len(results) =  0.6371235194585447 2247.387 591
0.6490673981191223 0.6371235194585447


In [ ]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 0.1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'softmax',
            'loss_batch': 128, 'loss_q': 0.98828125,
            'n': N
        }
    )
])

"""
=======
model =  CbKnn({'c': 0.1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'softmax', 'loss_batch': 128, 'loss_q': 0.98828125, 'n': 3000})
last loss =  tf.Tensor(-0.0034710548, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6589263322884013 3103.7275 1276
np.mean(results), mse, len(results) =  0.6371912013536379 3355.4568 591
0.6589263322884013 0.6371912013536379
"""

None

In [ ]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 0.1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'mse',
            # 'loss_batch': 128, 'loss_q': 0.98828125,
            'n': N
        }
    )
])

"""

=======
model =  CbKnn({'c': 0.1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'mse', 'n': 3000})
last loss =  tf.Tensor(0.30973396, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6398354231974922 0.30971116 1276
np.mean(results), mse, len(results) =  0.5676311336717429 0.82239026 591
0.6398354231974922 0.5676311336717429
"""

None

In [ ]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 0.1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'qmse',
            # 'loss_batch': 128, 'loss_q': 0.98828125,
            'n': N
        }
    )
])

"""
=======
model =  CbKnn({'c': 0.1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'qmse', 'n': 3000})
last loss =  tf.Tensor(0.33257723, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6397962382445141 0.33902717 1276
np.mean(results), mse, len(results) =  0.5576480541455162 0.8626492 591
0.6397962382445141 0.5576480541455162
"""

None

In [ ]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 0.1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'ApproxNDCGLoss',
            'loss_batch': 128, 'loss_q': 0.98828125,
            'n': N
        }
    )
])

"""
=======
model =  CbKnn({'c': 0.1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'ApproxNDCGLoss', 'loss_batch': 128, 'loss_q': 0.98828125, 'n': 3000})
last loss =  tf.Tensor(-0.90195936, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6618181818181817 8397.779 1276
np.mean(results), mse, len(results) =  0.6413028764805414 8809.353 591
0.6618181818181817 0.6413028764805414
"""

None

In [ ]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 0.1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'softmax',
            'loss_batch': 128, 'loss_q': 0.98828125,
            'n': N
        }
    )
])

"""
=======
model =  CbKnn({'c': 0.1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'softmax', 'loss_batch': 128, 'loss_q': 0.98828125, 'n': 3000})
last loss =  tf.Tensor(-0.0062250863, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6627194357366771 9964.674 1276
np.mean(results), mse, len(results) =  0.639103214890017 10727.211 591
0.6627194357366771 0.639103214890017
"""

None

In [48]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17).set_top_games_as_key(),
        fit_kwargs={
            'c': 0.1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'softmax_weighted',
            'loss_batch': 128, 'loss_q': 0.98828125,
            'n': N
        }
    )
])

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
3.043569729389357e-123
101 1276 591
self.embed_users['train'].shape =  (1276, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (1276, 101)



model =  CbKnn({'c': 0.1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'softmax_weighted', 'loss_batch': 128, 'loss_q': 0.98828125, 'n': 3000})
last loss =  tf.Tensor(-0.015804939, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.631512539184953 1395.9661 1276
np.mean(results), mse, len(results) =  0.6202199661590524 1491.4944 591
0.631512539184953 0.6202199661590524


In [50]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17).set_top_games_as_key(),
        fit_kwargs={
            'c': 0.1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'softmax',
            'loss_batch': 128, 'loss_q': 0.98828125,
            'n': N
        }
    )
])

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
3.043569729389357e-123
101 1276 591
self.embed_users['train'].shape =  (1276, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (1276, 101)



model =  CbKnn({'c': 0.1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'softmax', 'loss_batch': 128, 'loss_q': 0.98828125, 'n': 3000})
last loss =  tf.Tensor(-0.006542956, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6610266457680251 6463.049 1276
np.mean(results), mse, len(results) =  0.6302876480541455 6295.74 591
0.6610266457680251 0.6302876480541455


In [55]:
gc.collect()
ev([
    Popular(load(L, seed=17))
])

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
3.043569729389357e-123
101 1276 591



model =  <__main__.Popular object at 0x7fd907741dd8>
np.mean(results), mse, len(results) =  0.466935736677116 1.698370307357653 1276
np.mean(results), mse, len(results) =  0.46734348561759725 1.6313559499369625 591
0.466935736677116 0.46734348561759725


In [47]:
sorted(us.flatten())

[7.678519614e-05,
 8.168366912e-05,
 8.188247739e-05,
 8.261317998e-05,
 8.441858517e-05,
 8.575403626e-05,
 8.592195081e-05,
 8.607107156e-05,
 8.674758283e-05,
 8.688902744e-05,
 8.809386782e-05,
 8.841963427e-05,
 8.878375229e-05,
 8.885117131e-05,
 8.9037414e-05,
 9.016720287e-05,
 9.016791591e-05,
 9.031748777e-05,
 9.035744006e-05,
 9.036767005e-05,
 9.138890164e-05,
 9.204974776e-05,
 9.217234765e-05,
 9.230038995e-05,
 9.239183419e-05,
 9.263632091e-05,
 9.310299356e-05,
 9.343415877e-05,
 9.371079796e-05,
 9.375101217e-05,
 9.397199756e-05,
 9.479195433e-05,
 9.524959023e-05,
 9.543386841e-05,
 9.544780914e-05,
 9.546342335e-05,
 9.556538134e-05,
 9.581531776e-05,
 9.616909665e-05,
 9.658697672e-05,
 9.663745004e-05,
 9.674877219e-05,
 9.678734932e-05,
 9.686635894e-05,
 9.714973567e-05,
 9.735748608e-05,
 9.743600094e-05,
 9.751379548e-05,
 9.776975639e-05,
 9.788495663e-05,
 9.795866936e-05,
 9.802342538e-05,
 9.844369197e-05,
 9.851831419e-05,
 9.883131133e-05,
 9.897496784

In [ ]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 0.1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'softmax_weighted',
            'loss_batch': 64, 'loss_q': 0.5,
            'n': N
        }
    )
])

"""
=======
model =  CbKnn({'c': 0.1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'softmax_weighted', 'loss_batch': 64, 'loss_q': 0.5, 'n': 3000})
last loss =  tf.Tensor(-0.0047433823, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.623448275862069 3473.7312 1276
np.mean(results), mse, len(results) =  0.6147038917089678 3733.2517 591
0.623448275862069 0.6147038917089678
"""

None

In [ ]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 0.1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'softmax_weighted',
            'loss_batch': 256, 'loss_q': 0.5,
            'n': N
        }
    )
])

"""
=======
model =  CbKnn({'c': 0.1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'softmax_weighted', 'loss_batch': 64, 'loss_q': 0.5, 'n': 3000})
last loss =  tf.Tensor(-0.0047433823, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.623448275862069 3473.7312 1276
np.mean(results), mse, len(results) =  0.6147038917089678 3733.2517 591
0.623448275862069 0.6147038917089678
"""

None

In [ ]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 0.1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'softmax_weighted',
            'loss_batch': 256, 'loss_q': 0.9,
            'n': N
        }
    )
])

"""
=======
model =  CbKnn({'c': 0.1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'softmax_weighted', 'loss_batch': 256, 'loss_q': 0.9, 'n': 3000})
last loss =  tf.Tensor(-0.011163881, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.5897413793103449 1172.6676 1276
np.mean(results), mse, len(results) =  0.5752622673434856 1212.956 591
0.5897413793103449 0.5752622673434856
"""

None

In [ ]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 0.1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'softmax_weighted',
            'loss_batch': 256, 'loss_q': 0.99,
            'n': N
        }
    )
])

"""
=======
model =  CbKnn({'c': 0.1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'softmax_weighted', 'loss_batch': 256, 'loss_q': 0.99, 'n': 3000})
last loss =  tf.Tensor(-0.011156101, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6561520376175548 1160.4203 1276
np.mean(results), mse, len(results) =  0.6376480541455161 1250.4496 591
0.6561520376175548 0.6376480541455161
"""

None

In [ ]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 0.1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'softmax_weighted',
            'loss_batch': 1e9, 'loss_q': 0.99,
            'n': 100
        }
    )
])

"""
=======
model =  CbKnn({'c': 0.1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'softmax_weighted', 'loss_batch': 1000000000.0, 'loss_q': 0.99, 'n': 100})
last loss =  tf.Tensor(-0.0012877093, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.45856583072100315 175.49294 1276
np.mean(results), mse, len(results) =  0.467005076142132 191.04338 591
0.45856583072100315 0.467005076142132
"""

None

In [ ]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 0.1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'softmax',
            'loss_batch': 1e9, 'loss_q': 0.99,
            'n': 100
        }
    )
])

"""
=======
model =  CbKnn({'c': 0.1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'softmax', 'loss_batch': 1000000000.0, 'loss_q': 0.99, 'n': 100})
last loss =  tf.Tensor(-5.948538e-05, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.4814811912225706 82.84884 1276
np.mean(results), mse, len(results) =  0.4744331641285956 85.677574 591
0.4814811912225706 0.4744331641285956
"""

None

In [ ]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 0.1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'softmax',
            'loss_batch': 1e9, 'loss_q': 0.99,
            'n': 1000
        }
    )
])

"""
=======
model =  CbKnn({'c': 0.1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'softmax', 'loss_batch': 1000000000.0, 'loss_q': 0.99, 'n': 1000})
last loss =  tf.Tensor(-5.974792e-05, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.4472962382445141 61.354362 1276
np.mean(results), mse, len(results) =  0.4384940778341793 64.372284 591
0.4472962382445141 0.4384940778341793
"""

None

In [ ]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 0.1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'softmax',
            'loss_batch': 512, 'loss_q': 0.99,
            'n': 1000
        }
    )
])

"""
=======
model =  CbKnn({'c': 0.1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'softmax', 'loss_batch': 512, 'loss_q': 0.99, 'n': 1000})
last loss =  tf.Tensor(-0.0018599342, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6488244514106583 542.3041 1276
np.mean(results), mse, len(results) =  0.6191201353637902 568.6411 591
0.6488244514106583 0.6191201353637902
"""

None

In [ ]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 0.1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'ApproxNDCGLoss',
            'loss_batch': 512, 'loss_q': 0.99,
            'n': 1000
        }
    )
])

"""
ResourceExhaustedError
"""

In [ ]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 0.1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'ApproxNDCGLoss',
            'loss_batch': 128, 'loss_q': 0.99,
            'n': N
        }
    )
])

"""
=======
model =  CbKnn({'c': 0.1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'ApproxNDCGLoss', 'loss_batch': 128, 'loss_q': 0.99, 'n': 3000})
last loss =  tf.Tensor(-0.914229, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6707053291536049 8800.376946372078 1276
np.mean(results), mse, len(results) =  0.6474788494077833 9160.099873642986 591
0.6707053291536049 0.6474788494077833
"""

None

In [ ]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 0.1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'ApproxNDCGLoss',
            'loss_batch': 128, 'loss_q': 0.99,
            'n': N * 10
        }
    )
])

"""
=======
model =  CbKnn({'c': 0.1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'ApproxNDCGLoss', 'loss_batch': 128, 'loss_q': 0.99, 'n': 30000})
last loss =  tf.Tensor(-0.9344058, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.655987460815047 4911782.055411881 1276
np.mean(results), mse, len(results) =  0.642351945854484 5243894.970649532 591
0.655987460815047 0.642351945854484
"""

None

In [ ]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 0.1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'ApproxNDCGLoss',
            'loss_batch': 128, 'loss_q': 0.99,
            'n': N * 10
        }
    )
])

"""
=======
model =  CbKnn({'c': 0.1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'ApproxNDCGLoss', 'loss_batch': 128, 'loss_q': 0.99, 'n': 30000})
last loss =  tf.Tensor(-0.9191331, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6523510971786834 3576203.427713858 1276
np.mean(results), mse, len(results) =  0.6381895093062606 3573759.1551058786 591
0.6523510971786834 0.6381895093062606
"""

None

In [ ]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 0.1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'softmax',
            'loss_batch': 128, 'loss_q': 0.99,
            'n': N * 10
        }
    )
])

"""
=======
model =  CbKnn({'c': 0.1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'softmax', 'loss_batch': 128, 'loss_q': 0.99, 'n': 30000})
last loss =  tf.Tensor(-0.0065741455, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6305407523510972 25717434.356249034 1276
np.mean(results), mse, len(results) =  0.6102030456852793 28772905.41323205 591
0.6305407523510972 0.6102030456852793
"""

None

In [ ]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 0.1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'softmax_weighted',
            'loss_batch': 128, 'loss_q': 0.99,
            'n': N * 10
        }
    )
])

"""
=======
model =  CbKnn({'c': 0.1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'softmax_weighted', 'loss_batch': 128, 'loss_q': 0.99, 'n': 30000})

last loss =  tf.Tensor(-0.0154383825, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6389184952978056 1100169.1540628015 1276
np.mean(results), mse, len(results) =  0.6153976311336719 1220414.9485882977 591
0.6389184952978056 0.6153976311336719
"""

None

In [ ]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 0.1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'softmax_weighted',
            'loss_batch': 128, 'loss_q': 0.90,
            'n': N * 10
        }
    )
])

"""
=======
model =  CbKnn({'c': 0.1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'softmax_weighted', 'loss_batch': 128, 'loss_q': 0.9, 'n': 30000})
last loss =  tf.Tensor(-0.01592392, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6547178683385579 1069694.747274446 1276
np.mean(results), mse, len(results) =  0.6188155668358715 1256334.4344825153 591
0.6547178683385579 0.6188155668358715
"""

None

In [ ]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 0.1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'softmax_weighted',
            'loss_batch': 1024, 'loss_q': 0.99,
            'n': N
        }
    )
])

"""
=======
model =  CbKnn({'c': 0.1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'softmax_weighted', 'loss_batch': 1024, 'loss_q': 0.99, 'n': 3000})
last loss =  tf.Tensor(-0.0144993905, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6075548589341693 1241.4246269435419 1276
np.mean(results), mse, len(results) =  0.5765820642978003 1304.2759525502706 591
0.6075548589341693 0.5765820642978003
"""

None

In [ ]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 0.1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'softmax',
            'loss_batch': 1024, 'loss_q': 0.99,
            'n': N
        }
    )
])

"""
=======
model =  CbKnn({'c': 0.1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'softmax', 'loss_batch': 1024, 'loss_q': 0.99, 'n': 3000})
last loss =  tf.Tensor(-0.0009638205, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6245532915360502 2765.7343945763555 1276
np.mean(results), mse, len(results) =  0.5945516074450085 3020.1056772836546 591
0.6245532915360502 0.5945516074450085
"""

None

In [ ]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 0.1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'softmax',
            'loss_batch': 1024, 'loss_q': 0.995,
            'n': N
        }
    )
])

"""
=======
model =  CbKnn({'c': 0.1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'softmax', 'loss_batch': 1024, 'loss_q': 0.995, 'n': 3000})
last loss =  tf.Tensor(-0.0009149365, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6548275862068965 2327.767629162825 1276
np.mean(results), mse, len(results) =  0.6174619289340102 2526.9860566257526 591
0.6548275862068965 0.6174619289340102
"""

None

In [ ]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 0.1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'softmax',
            'loss_batch': 1024, 'loss_q': 0.995,
            'n': N * 2
        }
    )
])

"""
=======
model =  CbKnn({'c': 0.1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'softmax', 'loss_batch': 1024, 'loss_q': 0.995, 'n': 6000})
last loss =  tf.Tensor(-0.0009459086, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6573746081504702 20832.722370394087 1276
np.mean(results), mse, len(results) =  0.6173434856175973 23286.008170471978 591
0.6573746081504702 0.6173434856175973
"""

None

In [ ]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 0, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'softmax',
            'loss_batch': 1024, 'loss_q': 0.995,
            'n': N
        }
    )
])

"""
=======
model =  CbKnn({'c': 0, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'softmax', 'loss_batch': 1024, 'loss_q': 0.995, 'n': 3000})
last loss =  tf.Tensor(-0.00095466006, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6749686520376176 1022.1152530975477 1276
np.mean(results), mse, len(results) =  0.6491878172588833 1060.602320230028 591
0.6749686520376176 0.6491878172588833
"""

None

In [ ]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 0, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'ApproxNDCGLoss',
            'loss_batch': 128, 'loss_q': 0.99,
            'n': N
        }
    )
])

"""
=======
model =  CbKnn({'c': 0, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'ApproxNDCGLoss', 'loss_batch': 128, 'loss_q': 0.99, 'n': 3000})
last loss =  tf.Tensor(-0.9472088, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6701489028213166 7461.088120685631 1276
np.mean(results), mse, len(results) =  0.6467681895093063 7682.379284171121 591
0.6701489028213166 0.6467681895093063
"""

None

In [ ]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 0, 'train_dssm': True,
            'train_popbias': False, 'train_bias': True,
            'verbose': False, 'loss': 'ApproxNDCGLoss',
            'loss_batch': 128, 'loss_q': 0.99,
            'n': N
        }
    )
])

"""
=======
model =  CbKnn({'c': 0, 'train_dssm': True, 'train_popbias': False, 'train_bias': True, 'verbose': False, 'loss': 'ApproxNDCGLoss', 'loss_batch': 128, 'loss_q': 0.99, 'n': 3000})
last loss =  tf.Tensor(-0.9141045, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6649216300940439 7221.273129319215 1276
np.mean(results), mse, len(results) =  0.6346700507614212 7299.255945675214 591
0.6649216300940439 0.6346700507614212
"""

None

In [ ]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 0, 'train_dssm': True,
            'train_popbias': True, 'train_bias': False,
            'verbose': False, 'loss': 'ApproxNDCGLoss',
            'loss_batch': 128, 'loss_q': 0.99,
            'n': N
        }
    )
])

"""
=======
model =  CbKnn({'c': 0, 'train_dssm': True, 'train_popbias': True, 'train_bias': False, 'verbose': False, 'loss': 'ApproxNDCGLoss', 'loss_batch': 128, 'loss_q': 0.99, 'n': 3000})
last loss =  tf.Tensor(-0.9212466, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6770924764890283 9484.37321427242 1276
np.mean(results), mse, len(results) =  0.6457191201353638 9771.583461846758 591
0.6770924764890283 0.6457191201353638
"""

None

In [ ]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 0, 'train_dssm': True,
            'train_popbias': True, 'train_bias': False,
            'verbose': False, 'loss': 'softmax',
            'loss_batch': 128, 'loss_q': 0.99,
            'n': N
        }
    )
])

"""
=======
model =  CbKnn({'c': 0, 'train_dssm': True, 'train_popbias': True, 'train_bias': False, 'verbose': False, 'loss': 'softmax', 'loss_batch': 128, 'loss_q': 0.99, 'n': 3000})
last loss =  tf.Tensor(-0.0063318764, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6696159874608151 2578.060871076335 1276
np.mean(results), mse, len(results) =  0.645736040609137 2760.9398629814673 591
0.6696159874608151 0.645736040609137
"""

None

In [ ]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 0, 'train_dssm': True,
            'train_popbias': False, 'train_bias': True,
            'verbose': False, 'loss': 'softmax',
            'loss_batch': 128, 'loss_q': 0.99,
            'n': N
        }
    )
])

"""
=======
model =  CbKnn({'c': 0, 'train_dssm': True, 'train_popbias': False, 'train_bias': True, 'verbose': False, 'loss': 'softmax', 'loss_batch': 128, 'loss_q': 0.99, 'n': 3000})
last loss =  tf.Tensor(-0.006198782, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6688793103448276 4278.410874012454 1276
np.mean(results), mse, len(results) =  0.6394754653130288 4577.046584562315 591
0.6688793103448276 0.6394754653130288
"""

None

In [ ]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'softmax',
            'loss_batch': 128, 'loss_q': 0.99,
            'n': N
        }
    )
])

"""
=======
model =  CbKnn({'c': 1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'softmax', 'loss_batch': 128, 'loss_q': 0.99, 'n': 3000})
last loss =  tf.Tensor(-0.0055759684, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6338949843260188 27183.381450764613 1276
np.mean(results), mse, len(results) =  0.6039593908629441 28700.967395281506 591
0.6338949843260188 0.6039593908629441
"""

None

In [ ]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'softmax',
            'loss_batch': 128, 'loss_q': 0.99,
            'n': N * 10
        }
    )
])

"""
=======
model =  CbKnn({'c': 1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'softmax', 'loss_batch': 128, 'loss_q': 0.99, 'n': 30000})
last loss =  tf.Tensor(-0.006489137, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6124921630094043 64107069.81754361 1276
np.mean(results), mse, len(results) =  0.5934179357021997 74805093.09755436 591
0.6124921630094043 0.5934179357021997
"""

None

In [ ]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'ApproxNDCGLoss',
            'loss_batch': 128, 'loss_q': 0.99,
            'n': N
        }
    )
])

"""
=======
model =  CbKnn({'c': 1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'ApproxNDCGLoss', 'loss_batch': 128, 'loss_q': 0.99, 'n': 3000})
last loss =  tf.Tensor(-0.90922457, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6527899686520376 9117.354809548164 1276
np.mean(results), mse, len(results) =  0.6444839255499154 9881.352060888503 591
0.6527899686520376 0.6444839255499154
"""

None

In [ ]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'ApproxNDCGLoss',
            'loss_batch': 128, 'loss_q': 0.99,
            'n': N * 10
        }
    )
])

"""
=======
model =  CbKnn({'c': 1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'ApproxNDCGLoss', 'loss_batch': 128, 'loss_q': 0.99, 'n': 30000})
last loss =  tf.Tensor(-0.9340832, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6358699059561128 19158006.13685981 1276
np.mean(results), mse, len(results) =  0.6127241962774957 20199543.076208066 591
0.6358699059561128 0.6127241962774957
"""

None

In [ ]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'ApproxNDCGLoss',
            'loss_batch': 128, 'loss_q': 0.99,
            'n': N,
            'ubatch': 1e9
        }
    )
])

"""
=======
model =  CbKnn({'c': 1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'ApproxNDCGLoss', 'loss_batch': 128, 'loss_q': 0.99, 'n': 3000, 'ubatch': 1000000000.0})
last loss =  tf.Tensor(-0.91337925, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6475235109717868 10695.207230147087 1276
np.mean(results), mse, len(results) =  0.6316243654822334 10954.29451080161 591
0.6475235109717868 0.6316243654822334
"""

None

In [ ]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'ApproxNDCGLoss',
            'loss_batch': 128, 'loss_q': 0.99,
            'n': N,
            'ubatch': 64
        }
    )
])

"""
=======
model =  CbKnn({'c': 1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'ApproxNDCGLoss', 'loss_batch': 128, 'loss_q': 0.99, 'n': 3000, 'ubatch': 64})
last loss =  tf.Tensor(-0.89977616, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.5782680250783699 84753.32394284962 1276
np.mean(results), mse, len(results) =  0.5685786802030457 94663.30066974477 591
0.5782680250783699 0.5685786802030457
"""

None

In [ ]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'ApproxNDCGLoss',
            'loss_batch': 128, 'loss_q': 0.99,
            'n': N * 10,
            'ubatch': 64
        }
    )
])

"""
=======
model =  CbKnn({'c': 1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'ApproxNDCGLoss', 'loss_batch': 128, 'loss_q': 0.99, 'n': 30000, 'ubatch': 64})
last loss =  tf.Tensor(-0.91063315, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.5488636363636363 126276071.032867 1276
np.mean(results), mse, len(results) =  0.5426057529610829 133288569.64100815 591
0.5488636363636363 0.5426057529610829
"""

None

In [ ]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'ApproxNDCGLoss',
            'loss_batch': 128, 'loss_q': 0.99,
            'n': N,
            'ubatch': 256
        }
    )
])


"""
=======
model =  CbKnn({'c': 1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'ApproxNDCGLoss', 'loss_batch': 128, 'loss_q': 0.99, 'n': 3000, 'ubatch': 256})
last loss =  tf.Tensor(-0.8927802, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6047178683385579 19796.720792345186 1276
np.mean(results), mse, len(results) =  0.6001861252115059 20414.410158103536 591
0.6047178683385579 0.6001861252115059
"""

None

In [ ]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'softmax',
            'loss_batch': 128, 'loss_q': 0.99,
            'n': N,
            'ubatch': 256
        }
    )
])

"""
=======
model =  CbKnn({'c': 1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'softmax', 'loss_batch': 128, 'loss_q': 0.99, 'n': 3000, 'ubatch': 256})
last loss =  tf.Tensor(-0.0072694337, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.5947100313479624 86974.36291148805 1276
np.mean(results), mse, len(results) =  0.5753299492385786 96504.21540956276 591
0.5947100313479624 0.5753299492385786
"""

None

In [ ]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'softmax',
            'loss_batch': 1e9, 'loss_q': 0.99,
            'n': N,
            'ubatch': 256
        }
    )
])

"""
=======
model =  CbKnn({'c': 1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'softmax', 'loss_batch': 1000000000.0, 'loss_q': 0.99, 'n': 3000, 'ubatch': 256})
last loss =  tf.Tensor(-6.031814e-05, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.46343260188087776 115.73264728059762 1276
np.mean(results), mse, len(results) =  0.4606768189509306 112.52010405570456 591
0.46343260188087776 0.4606768189509306
"""

None

In [12]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'softmax',
            'loss_batch': 1e9, 'loss_q': 0.99,
            'n': N,
            'ubatch': 512
        }
    )
])

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  1.4029005834692048e-113
1.4029005834692048e-113
101 1276 591
self.embed_users['train'].shape =  (1276, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (1276, 101)



model =  CbKnn({'c': 1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'softmax', 'loss_batch': 1000000000.0, 'loss_q': 0.99, 'n': 3000, 'ubatch': 512})
last loss =  tf.Tensor(-5.9490245e-05, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.49323667711598757 170.92165932803525 1276
np.mean(results), mse, len(results) =  0.4822842639593909 161.46228473015896 591
0.49323667711598757 0.4822842639593909


In [30]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': True, 'loss': 'sigmoid_top_100',
            'loss_batch': 1e9, 'loss_q': 0.99,
            "ubatch": 64,
            'n': 300,
        }
    )
])

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  1.4029005834692048e-113
1.4029005834692048e-113
101 1276 591
self.embed_users['train'].shape =  (1276, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (1276, 101)



model =  CbKnn({'c': 1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': True, 'loss': 'sigmoid_top_100', 'loss_batch': 1000000000.0, 'loss_q': 0.99, 'ubatch': 64, 'n': 300})
self.W =  <tf.Variable 'Variable:0' shape=(100, 101) dtype=float32, numpy=
array([[ 1.7122873e-03,  1.5744284e-03,  4.1468443e-06, ...,
         9.8867330e-04,  1.2266752e-03, -1.0771827e-03],
       [ 1.7162145e-03,  1.7601653e-04, -6.1627179e-05, ...,
         4.8833190e-05,  9.3180686e-05, -1.0102660e-03],
       [-1.1734755e-03, -1.0295276e-03, -7.8503886e-04, ...,
        -7.6046342e-04,  1.1493116e-05,  1.2722349e-03],


In [29]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 0.1, 'train_dssm': False,
            'train_popbias': False, 'train_bias': False,
            'verbose': True, 'loss': 'sigmoid_top_100',
            'loss_batch': 1e9, 'loss_q': 0.99,
            "ubatch": 64,
            'n': 300,
        }
    )
])

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  1.4029005834692048e-113
1.4029005834692048e-113
101 1276 591
self.embed_users['train'].shape =  (1276, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (1276, 101)



model =  CbKnn({'c': 0.1, 'train_dssm': False, 'train_popbias': False, 'train_bias': False, 'verbose': True, 'loss': 'sigmoid_top_100', 'loss_batch': 1000000000.0, 'loss_q': 0.99, 'ubatch': 64, 'n': 300})
self.W =  <tf.Variable 'Variable:0' shape=(100, 101) dtype=float32, numpy=
array([[ 3.4934044e-04, -7.2420336e-04, -5.3568609e-04, ...,
        -9.3042140e-04, -1.0921023e-03,  4.9959944e-04],
       [-1.3499215e-04, -1.2631597e-03, -3.3104033e-04, ...,
         1.0737139e-03,  1.4029455e-03,  8.6479576e-04],
       [ 2.1068379e-04,  8.1237796e-04,  6.0749607e-04, ...,
         8.5498694e-05,  3.3382326e-05,  1.2401014e-03

In [41]:
1 - 2 * (np.arange(5) < 2)

array([-1, -1,  1,  1,  1])

In [31]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 0.1, 'train_dssm': False,
            'train_popbias': False, 'train_bias': False,
            'verbose': True, 'loss': 'sigmoid_top_100',
            'loss_batch': 1e9, 'loss_q': 0.99,
            "ubatch": 128,
            'n': 300,
        }
    )
])

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  1.4029005834692048e-113
1.4029005834692048e-113
101 1276 591
self.embed_users['train'].shape =  (1276, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (1276, 101)



model =  CbKnn({'c': 0.1, 'train_dssm': False, 'train_popbias': False, 'train_bias': False, 'verbose': True, 'loss': 'sigmoid_top_100', 'loss_batch': 1000000000.0, 'loss_q': 0.99, 'ubatch': 128, 'n': 300})
self.W =  <tf.Variable 'Variable:0' shape=(100, 101) dtype=float32, numpy=
array([[-4.8171365e-04, -8.1955112e-04,  1.6699028e-03, ...,
        -1.5156337e-03,  1.5832737e-04,  1.2851686e-03],
       [ 6.0203031e-04, -1.6523078e-04,  8.3850860e-04, ...,
        -7.5644063e-04, -1.4386349e-03, -4.0499435e-04],
       [-1.1182777e-03, -7.8003970e-04,  8.1391959e-04, ...,
         4.1695999e-04,  1.5434098e-03, -4.8619829e-0

In [35]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 0.1, 'train_dssm': False,
            'train_popbias': True, 'train_bias': False,
            'verbose': True, 'loss': 'sigmoid_top_100',
            'loss_batch': 1e9, 'loss_q': 0.99,
            # "ubatch": 128,
            'n': 30,
        }
    )
])

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  1.4029005834692048e-113
1.4029005834692048e-113
101 1276 591
self.embed_users['train'].shape =  (1276, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (1276, 101)



model =  CbKnn({'c': 0.1, 'train_dssm': False, 'train_popbias': True, 'train_bias': False, 'verbose': True, 'loss': 'sigmoid_top_100', 'loss_batch': 1000000000.0, 'loss_q': 0.99, 'n': 30})
self.W =  <tf.Variable 'Variable:0' shape=(100, 101) dtype=float32, numpy=
array([[ 1.0317180e-03, -3.1516387e-04,  1.4656853e-03, ...,
        -4.5293703e-04,  7.2105048e-04,  9.1890246e-04],
       [-4.8986523e-05, -6.7557278e-04,  2.6456191e-04, ...,
         7.5561181e-04, -9.9252397e-04, -3.2907783e-04],
       [ 5.4117007e-04,  2.0795227e-04, -9.0920529e-04, ...,
         1.0694510e-03,  7.8640314e-04,  1.6283643e-03],
       ...,
 

In [36]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 0.1, 'train_dssm': False,
            'train_popbias': True, 'train_bias': False,
            'verbose': True, 'loss': 'sigmoid_top_100',
            'loss_batch': 1e9, 'loss_q': 0.99,
            # "ubatch": 128,
            'n': 100,
        }
    )
])

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  1.4029005834692048e-113
1.4029005834692048e-113
101 1276 591
self.embed_users['train'].shape =  (1276, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (1276, 101)



model =  CbKnn({'c': 0.1, 'train_dssm': False, 'train_popbias': True, 'train_bias': False, 'verbose': True, 'loss': 'sigmoid_top_100', 'loss_batch': 1000000000.0, 'loss_q': 0.99, 'n': 100})
self.W =  <tf.Variable 'Variable:0' shape=(100, 101) dtype=float32, numpy=
array([[-1.50741532e-03, -1.57222603e-04,  6.20788487e-04, ...,
        -9.30914131e-04,  1.65616092e-03,  8.30048302e-05],
       [-7.31218839e-04, -9.78634343e-04, -1.13986134e-04, ...,
         1.17891934e-03,  5.81895409e-04,  1.67974201e-03],
       [ 5.35400934e-04, -1.71634089e-03,  1.56147336e-03, ...,
        -1.47732545e-03,  1.47716375e-03, -6.61892758e

In [43]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 0.1, 'train_dssm': False,
            'train_popbias': True, 'train_bias': False,
            'verbose': True, 'loss': 'sigmoid_top_100',
            'loss_batch': 1e9, 'loss_q': 0.99,
            # "ubatch": 128,
            'n': 100,
        }
    )
])

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  1.4029005834692048e-113
1.4029005834692048e-113
101 1276 591
self.embed_users['train'].shape =  (1276, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (1276, 101)



model =  CbKnn({'c': 0.1, 'train_dssm': False, 'train_popbias': True, 'train_bias': False, 'verbose': True, 'loss': 'sigmoid_top_100', 'loss_batch': 1000000000.0, 'loss_q': 0.99, 'n': 100})
self.W =  <tf.Variable 'Variable:0' shape=(100, 101) dtype=float32, numpy=
array([[-1.6234335e-03, -1.0509230e-03,  6.8351510e-04, ...,
        -8.2457619e-04, -4.9722753e-04,  1.5453801e-04],
       [ 3.4458027e-04,  1.0742602e-03,  1.6149432e-05, ...,
         9.9591492e-04, -3.5855232e-04, -1.5611080e-03],
       [-1.2425326e-03, -2.1844938e-04, -5.5330037e-04, ...,
        -1.0531857e-03,  4.6293362e-04, -5.7642133e-04],
       ...,


In [44]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 0.1, 'train_dssm': True,
            'train_popbias': True, 'train_bias': False,
            'verbose': True, 'loss': 'sigmoid_top_100',
            'loss_batch': 1e9, 'loss_q': 0.99,
            # "ubatch": 128,
            'n': 100,
        }
    )
])

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  1.4029005834692048e-113
1.4029005834692048e-113
101 1276 591
self.embed_users['train'].shape =  (1276, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (1276, 101)



model =  CbKnn({'c': 0.1, 'train_dssm': True, 'train_popbias': True, 'train_bias': False, 'verbose': True, 'loss': 'sigmoid_top_100', 'loss_batch': 1000000000.0, 'loss_q': 0.99, 'n': 100})
self.W =  <tf.Variable 'Variable:0' shape=(100, 101) dtype=float32, numpy=
array([[-1.1238138e-03,  3.4361065e-04, -1.2237697e-03, ...,
         4.1613489e-04,  1.3963884e-03, -1.3741207e-03],
       [ 2.5116206e-05,  1.1658132e-03,  5.4697692e-04, ...,
        -1.6617290e-03, -6.8013369e-06, -1.6837046e-04],
       [ 7.7261927e-04,  9.0711832e-04, -6.6123327e-04, ...,
         1.3113738e-04, -1.5481275e-03, -1.5261481e-03],
       ...,
 

In [45]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 0.1, 'train_dssm': True,
            'train_popbias': True, 'train_bias': False,
            'verbose': True, 'loss': 'sigmoid_top_100',
            'loss_batch': 1e9, 'loss_q': 0.99,
            # "ubatch": 128,
            'n': 100,
            "learning_rate": 0.01,
        }
    )
])

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  1.4029005834692048e-113
1.4029005834692048e-113
101 1276 591
self.embed_users['train'].shape =  (1276, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (1276, 101)



model =  CbKnn({'c': 0.1, 'train_dssm': True, 'train_popbias': True, 'train_bias': False, 'verbose': True, 'loss': 'sigmoid_top_100', 'loss_batch': 1000000000.0, 'loss_q': 0.99, 'n': 100, 'learning_rate': 0.01})


KeyboardInterrupt: 

In [49]:
L

2000

In [23]:
gc.collect()
ev([
    CBKnnV0(load(3000, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'ApproxNDCGLoss',
            'loss_batch': 128, 'loss_q': 0.99,
            'n': 20000 # N,
            # 'ubatch': 256
        }
    )
])

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  2.0365795563173995e-113
2.0365795563173995e-113
101 1967 887
self.embed_users['train'].shape =  (1967, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (1967, 101)



model =  CbKnn({'c': 1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'ApproxNDCGLoss', 'loss_batch': 128, 'loss_q': 0.99, 'n': 20000})
last loss =  tf.Tensor(-0.93994176, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6703762074224708 1184077.063722335 1967
np.mean(results), mse, len(results) =  0.6482638105975197 1304564.9783255665 887
0.6703762074224708 0.6482638105975197


In [24]:
gc.collect()
ev([
    CBKnnV0(load(3000, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 1, 'train_dssm': True,
            'train_popbias': True, 'train_bias': False,
            'verbose': False, 'loss': 'ApproxNDCGLoss',
            'loss_batch': 128, 'loss_q': 0.99,
            'n': 20000 # N,
            # 'ubatch': 256
        }
    )
])

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  2.0365795563173995e-113
2.0365795563173995e-113
101 1967 887
self.embed_users['train'].shape =  (1967, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (1967, 101)



model =  CbKnn({'c': 1, 'train_dssm': True, 'train_popbias': True, 'train_bias': False, 'verbose': False, 'loss': 'ApproxNDCGLoss', 'loss_batch': 128, 'loss_q': 0.99, 'n': 20000})
last loss =  tf.Tensor(-0.94859225, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6720589730554144 1218170.3699426944 1967
np.mean(results), mse, len(results) =  0.6482187147688839 1281280.0897754305 887
0.6720589730554144 0.6482187147688839


In [25]:
gc.collect()
ev([
    CBKnnV0(load(3000, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 0, 'train_dssm': True,
            'train_popbias': True, 'train_bias': False,
            'verbose': False, 'loss': 'ApproxNDCGLoss',
            'loss_batch': 128, 'loss_q': 0.99,
            'n': 20000 # N,
            # 'ubatch': 256
        }
    )
])

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  2.0365795563173995e-113
2.0365795563173995e-113
101 1967 887
self.embed_users['train'].shape =  (1967, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (1967, 101)



model =  CbKnn({'c': 0, 'train_dssm': True, 'train_popbias': True, 'train_bias': False, 'verbose': False, 'loss': 'ApproxNDCGLoss', 'loss_batch': 128, 'loss_q': 0.99, 'n': 20000})
last loss =  tf.Tensor(-0.952636, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.695521098118963 368530.4971283056 1967
np.mean(results), mse, len(results) =  0.6712965050732808 393198.47271001723 887
0.695521098118963 0.6712965050732808


In [13]:
gc.collect()
ev([
    CBKnnV0(load(3000, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'ApproxNDCGLoss',
            'loss_batch': 128, 'loss_q': 0.99,
            'n': 10000 # N,
            # 'ubatch': 256
        }
    )
])

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  2.0365795563173995e-113
2.0365795563173995e-113
101 1967 887
self.embed_users['train'].shape =  (1967, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (1967, 101)



model =  CbKnn({'c': 1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'ApproxNDCGLoss', 'loss_batch': 128, 'loss_q': 0.99, 'n': 10000})
last loss =  tf.Tensor(-0.9538621, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6830299949161159 93505.7531056077 1967
np.mean(results), mse, len(results) =  0.6601578354002254 102628.79643057271 887
0.6830299949161159 0.6601578354002254


In [18]:
gc.collect()
ev([
    DssmKnn(load(3000, seed=17, det_attempts=50000).set_top_games_as_key(), 6,
        fit_kwargs={
            'c': 1, 'train_dssm': True,
            'train_popbias': True, 'train_bias': False,
            'verbose': False, 'loss': 'ApproxNDCGLoss',
            'loss_batch': 128, 'loss_q': 0.99,
            'n': 10000 # N,
            # 'ubatch': 256
        }
    )
])

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  2.0365795563173995e-113
2.0365795563173995e-113
101 1967 887
self.embed_users['train'].shape =  (1967, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (1967, 101)



model =  DssmKnn(6,{'c': 1, 'train_dssm': True, 'train_popbias': True, 'train_bias': False, 'verbose': False, 'loss': 'ApproxNDCGLoss', 'loss_batch': 128, 'loss_q': 0.99, 'n': 10000})
last loss =  tf.Tensor(-0.96127576, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6346822572445349 7.549674844837284 1967
np.mean(results), mse, len(results) =  0.6183990980834273 7.407633059102243 887
0.6346822572445349 0.6183990980834273


In [19]:
gc.collect()
ev([
    DssmKnn(load(3000, seed=17, det_attempts=50000).set_top_games_as_key(), 6,
        fit_kwargs={
            'c': 1, 'train_dssm': True,
            'train_popbias': True, 'train_bias': False,
            'verbose': False, 'loss': 'softmax',
            'loss_batch': 128, 'loss_q': 0.99,
            'n': 10000 # N,
            # 'ubatch': 256
        }
    )
])

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  2.0365795563173995e-113
2.0365795563173995e-113
101 1967 887
self.embed_users['train'].shape =  (1967, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (1967, 101)



model =  DssmKnn(6,{'c': 1, 'train_dssm': True, 'train_popbias': True, 'train_bias': False, 'verbose': False, 'loss': 'softmax', 'loss_batch': 128, 'loss_q': 0.99, 'n': 10000})
last loss =  tf.Tensor(-0.006700675, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6661260803253686 129.7650575989331 1967
np.mean(results), mse, len(results) =  0.6542051860202931 128.85949769314482 887
0.6661260803253686 0.6542051860202931


In [22]:
gc.collect()
ev([
    DssmKnn(load(3000, seed=17, det_attempts=50000).set_top_games_as_key(), 6,
        fit_kwargs={
            'c': 1, 'train_dssm': True,
            'train_popbias': True, 'train_bias': False,
            'verbose': False, 'loss': 'softmax',
            'loss_batch': 128, 'loss_q': 0.99,
            'n': 20000 # N,
            # 'ubatch': 256
        }
    )
])

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  2.0365795563173995e-113
2.0365795563173995e-113
101 1967 887
self.embed_users['train'].shape =  (1967, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (1967, 101)



model =  DssmKnn(6,{'c': 1, 'train_dssm': True, 'train_popbias': True, 'train_bias': False, 'verbose': False, 'loss': 'softmax', 'loss_batch': 128, 'loss_q': 0.99, 'n': 20000})
last loss =  tf.Tensor(-0.006017455, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.7111235383833249 150.33590728143878 1967
np.mean(results), mse, len(results) =  0.6968996617812852 147.41773816034342 887
0.7111235383833249 0.6968996617812852


In [20]:
gc.collect()
ev([
    CBKnnV0(load(3000, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'softmax',
            'loss_batch': 128, 'loss_q': 0.99,
            'n': 10000 # N,
            # 'ubatch': 256
        }
    )
])

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  2.0365795563173995e-113
2.0365795563173995e-113
101 1967 887
self.embed_users['train'].shape =  (1967, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (1967, 101)



model =  CbKnn({'c': 1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'softmax', 'loss_batch': 128, 'loss_q': 0.99, 'n': 10000})
last loss =  tf.Tensor(-0.0067253886, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6660091509913574 480336.5215417338 1967
np.mean(results), mse, len(results) =  0.6311048478015783 518929.80843886506 887
0.6660091509913574 0.6311048478015783


In [21]:
gc.collect()
ev([
    CBKnnV0(load(3000, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 1, 'train_dssm': True,
            'train_popbias': True, 'train_bias': False,
            'verbose': False, 'loss': 'softmax',
            'loss_batch': 128, 'loss_q': 0.99,
            'n': 10000 # N,
            # 'ubatch': 256
        }
    )
])

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  2.0365795563173995e-113
2.0365795563173995e-113
101 1967 887
self.embed_users['train'].shape =  (1967, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (1967, 101)



model =  CbKnn({'c': 1, 'train_dssm': True, 'train_popbias': True, 'train_bias': False, 'verbose': False, 'loss': 'softmax', 'loss_batch': 128, 'loss_q': 0.99, 'n': 10000})
last loss =  tf.Tensor(-0.0059691947, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6666192170818506 521647.20974943676 1967
np.mean(results), mse, len(results) =  0.6468432919954904 562970.1074089524 887
0.6666192170818506 0.6468432919954904


In [17]:
gc.collect()
ev([
    DssmKnn(load(3000, seed=17, det_attempts=50000).set_top_games_as_key(), 6,
        fit_kwargs={
            'c': 1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'ApproxNDCGLoss',
            'loss_batch': 128, 'loss_q': 0.99,
            'n': 10000 # N,
            # 'ubatch': 256
        }
    )
])

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  2.0365795563173995e-113
2.0365795563173995e-113
101 1967 887
self.embed_users['train'].shape =  (1967, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (1967, 101)



model =  DssmKnn(6,{'c': 1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'ApproxNDCGLoss', 'loss_batch': 128, 'loss_q': 0.99, 'n': 10000})
last loss =  tf.Tensor(-0.9493202, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.5905439755973564 833.6285362980228 1967
np.mean(results), mse, len(results) =  0.5845208568207441 810.7264040335976 887
0.5905439755973564 0.5845208568207441


In [8]:
gc.collect()
ev([
    CBKnnV0(load(3000, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'ApproxNDCGLoss',
            'loss_batch': 128, 'loss_q': 0.99,
            'n': N,
            # 'ubatch': 256
        }
    )
])

WARN: buffered limit is different, 2000 != 3000, reloading...


[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  2.0365795563173995e-113
2.0365795563173995e-113
101 1967 887
self.embed_users['train'].shape =  (1967, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (1967, 101)



model =  CbKnn({'c': 1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'ApproxNDCGLoss', 'loss_batch': 128, 'loss_q': 0.99, 'n': 3000})
last loss =  tf.Tensor(-0.9348084, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6744738179969497 4742.551430266089 1967
np.mean(results), mse, len(results) =  0.6545208568207441 5090.890486659378 887
0.6744738179969497 0.6545208568207441


In [9]:
gc.collect()
ev([
    DssmKnn(load(3000, seed=17, det_attempts=50000).set_top_games_as_key(), 6,
        fit_kwargs={
            'c': 1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'ApproxNDCGLoss',
            'loss_batch': 128, 'loss_q': 0.99,
            'n': N,
            # 'ubatch': 256
        }
    )
])

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  2.0365795563173995e-113
2.0365795563173995e-113
101 1967 887
self.embed_users['train'].shape =  (1967, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (1967, 101)



model =  DssmKnn(6,{'c': 1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'ApproxNDCGLoss', 'loss_batch': 128, 'loss_q': 0.99, 'n': 3000})
last loss =  tf.Tensor(-0.74748653, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.5078952719877987 217.8242866407207 1967
np.mean(results), mse, len(results) =  0.5012626832018039 211.797307477917 887
0.5078952719877987 0.5012626832018039


In [10]:
gc.collect()
ev([
    DssmKnn(load(2000, seed=17, det_attempts=50000).set_top_games_as_key(), 6,
        fit_kwargs={
            'c': 1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'ApproxNDCGLoss',
            'loss_batch': 128, 'loss_q': 0.99,
            'n': N,
            # 'ubatch': 256
        }
    )
])

WARN: buffered limit is different, 3000 != 2000, reloading...


[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  1.4029005834692048e-113
1.4029005834692048e-113
101 1276 591
self.embed_users['train'].shape =  (1276, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (1276, 101)



model =  DssmKnn(6,{'c': 1, 'train_dssm': True, 'train_popbias': False, 'train_bias': False, 'verbose': False, 'loss': 'ApproxNDCGLoss', 'loss_batch': 128, 'loss_q': 0.99, 'n': 3000})
last loss =  tf.Tensor(-0.83495253, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.49986677115987455 80.00359940965872 1276
np.mean(results), mse, len(results) =  0.5013705583756346 79.5289401895007 591
0.49986677115987455 0.5013705583756346


In [11]:
gc.collect()
ev([
    DssmKnn(load(2000, seed=17, det_attempts=50000).set_top_games_as_key(), 6,
        fit_kwargs={
            'c': 1, 'train_dssm': True,
            'train_popbias': True, 'train_bias': False,
            'verbose': False, 'loss': 'ApproxNDCGLoss',
            'loss_batch': 128, 'loss_q': 0.99,
            'n': N,
            # 'ubatch': 256
        }
    )
])

[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  1.4029005834692048e-113
1.4029005834692048e-113
101 1276 591
self.embed_users['train'].shape =  (1276, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (1276, 101)



model =  DssmKnn(6,{'c': 1, 'train_dssm': True, 'train_popbias': True, 'train_bias': False, 'verbose': False, 'loss': 'ApproxNDCGLoss', 'loss_batch': 128, 'loss_q': 0.99, 'n': 3000})
last loss =  tf.Tensor(-0.91764086, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.631191222570533 2.6829214517290945 1276
np.mean(results), mse, len(results) =  0.6255160744500846 2.9145671512007154 591
0.631191222570533 0.6255160744500846


In [12]:
gc.collect()
ev([
    DssmKnn(load(3000, seed=17, det_attempts=50000).set_top_games_as_key(), 6,
        fit_kwargs={
            'c': 1, 'train_dssm': True,
            'train_popbias': True, 'train_bias': False,
            'verbose': False, 'loss': 'ApproxNDCGLoss',
            'loss_batch': 128, 'loss_q': 0.99,
            'n': N,
            # 'ubatch': 256
        }
    )
])

WARN: buffered limit is different, 2000 != 3000, reloading...


[(6, 16514), (7, 16514), (8, 16514), (9, 16514)]
Best det =  2.0365795563173995e-113
2.0365795563173995e-113
101 1967 887
self.embed_users['train'].shape =  (1967, 100)
self.embed_games.shape =  (16514, 101)
self.games2users.shape =  (100, 101)
self.core_users_scores.shape =  (101, 16514)
self.core_users_embs.shape =  (101, 100)
self.ge_users.shape =  (1967, 101)



model =  DssmKnn(6,{'c': 1, 'train_dssm': True, 'train_popbias': True, 'train_bias': False, 'verbose': False, 'loss': 'ApproxNDCGLoss', 'loss_batch': 128, 'loss_q': 0.99, 'n': 3000})
last loss =  tf.Tensor(-0.87075037, shape=(), dtype=float32)
np.mean(results), mse, len(results) =  0.6105897305541433 3.031786586347196 1967
np.mean(results), mse, len(results) =  0.5967080045095828 2.745402738240331 887
0.6105897305541433 0.5967080045095828


In [ ]:
gc.collect()
ev([
    CBKnnV0(load(3000, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'ApproxNDCGLoss',
            'loss_batch': 128, 'loss_q': 0.99,
            'n': N,
            # 'ubatch': 256
        }
    )
])

In [ ]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 0.1, 'train_dssm': False,
            'train_popbias': True, 'train_bias': False,
            'verbose': True, 'loss': 'sigmoid_top_100',
            'loss_batch': 1e9, 'loss_q': 0.99,
            # "ubatch": 128,
            'n': 100,
        }
    )
])

In [ ]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'softmax',
            'loss_batch': 1e9, 'loss_q': 0.99,
            'n': N * 3,
            'ubatch': 512
        }
    )
])

In [ ]:
gc.collect()
ev([
    CBKnnV0(load(L, seed=17, det_attempts=50000).set_top_games_as_key(),
        fit_kwargs={
            'c': 1, 'train_dssm': True,
            'train_popbias': False, 'train_bias': False,
            'verbose': False, 'loss': 'qrmse',
            'loss_batch': 1e9, 'loss_q': 0.99,
            'n': N * 3,
            'ubatch': 512
        }
    )
])

TBD - семплирование строк
- CE
- MAX Pooling
- all users

In [ ]:
-Лучшие ключи в терминах векторов таргетов
- итеративый их выбор
- разложение матрицы